<a href="https://colab.research.google.com/github/sherjahong1r/Machine-Learning-Lessons/blob/main/Copy_of_13_Shug'illantirilgan_modelni_moslashtirsh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Modelni moslashtirish**

## **Avval shug'ullantirilgan modelni o'z ma'lumotimizga moslashtirish**

In [ ]:
!pip install -U datasets fsspec
# Bu buyruq datasets va fsspec kutubxonalarini o'rnatadi yoki ularni eng so'nggi versiyaga yangilaydi.
# datasets ma'lumotlar to'plamlari bilan ishlash uchun, fsspec esa turli fayl tizimlari bilan bog'lanish uchun ishlatiladi.
# -U (yoki --upgrade) belgisi esa, agar bu kutubxonalar allaqachon o'rnatilgan bo'lsa, ularni eng so'nggi mavjud versiyasiga yangilashni bildiradi.

In [ ]:
!pip install -U transformers
# Bu  buyrug'i transformers kutubxonasini o'rnatadi yoki uni eng so'nggi versiyasiga yangilaydi. transformers - bu
# Hugging Face kompaniyasining kutubxonasi bo'lib, u tayyorlangan (pre-trained) modellar, masalan, BERT, GPT-2, T5 kabi
# modellarni yuklash, ulardan foydalanish va o'z ma'lumotlaringizga moslashtirish (finetuning) uchun ishlatiladi.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
small_train = dataset["train"].shuffle(seed=42).select([1 for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([1 for i in list(range(300))])

# Bu kod datasets kutubxonasidan foydalanib "imdb" ma'lumotlar to'plamini yuklaydi. So'ngra,
# u o'qitish uchun 1000 ta va testlash uchun 300 ta misoldan iborat kichik qismlarni tasodifiy tanlab oladi.
# Bu modelni tezroq sinovdan o'tkazish uchun mo'ljallangan.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

# Bu kod AutoTokenizer yordamida distilbert-base-uncased modeliga mos keladigan tokenizatorni yuklaydi.
# preprocess_function esa matnni tokenizatsiya qilish, qirqish (truncation) va to'ldirish (padding) orqali
# modelga kiritish uchun tayyorlaydi. Yakunda, bu funksiya small_train va small_test ma'lumotlar to'plamlariga
# qo'llanilib, ularni tokenizatsiyalangan formatga o'tkazadi.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Bu kod transformers kutubxonasidan AutoModelForSequenceClassification klassini import qiladi va distilbert-base-uncased
# modelini ikki sinf (num_labels=2) uchun moslashtirilgan tarzda yuklaydi. Bu odatda matn tasniflash vazifalari uchun
# tayyorlangan modelni ishga tushirishni anglatadi.
# num_labels=2 bu modelning qancha chiqish sinflariga ega bo'lishini bildiradi. Bizning holatimizda, IMDB ma'lumotlar to'plami
# bilan ishlayapmiz, bu erda har bir kinofilm sharhi ikki toifadan biriga bo'linadi: ijobiy (positive) yoki salbiy (negative).
# 0 (salbiy) - sharh salbiy fikrni bildiradi.
# 1 (ijobiy) - sharh ijobiy fikrni bildiradi.
# Shuning uchun, model sharhlarni ikkita guruhga ajratishi kerak, ya'ni ikkita "label" yoki "sinf" mavjud. Shu sababli,
# num_labels qiymati 2 ga teng qilib belgilangan. Agar siz ko'proq sinflarga ega bo'lgan boshqa turdagi tasniflash vazifasi
# bilan ishlayotgan bo'lsangiz, bu qiymat o'zgarishi mumkin edi (masalan, uchta sinf bo'lsa num_labels=3 bo'ladi).

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=10,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()
# training_args = TrainingArguments(...): Bu erda modelni o'qitish uchun turli xil argumentlar (TrainingArguments) o'rnatiladi:
# output_dir="./results": O'qitish natijalari (masalan, modelning saqlangan versiyalari, loglar) saqlanadigan katalog.
# num_train_epochs=3: Model qancha epoxa davomida o'qitilishini belgilaydi (bu holatda 3 marta).
# per_device_train_batch_size=16: Har bir qurilma (GPU) uchun o'qitish partiyasining hajmi.
# per_device_eval_batch_size=16: Har bir qurilma uchun baholash partiyasining hajmi.
# logging_steps=10: Loglar (o'qitish jarayoni haqidagi ma'lumotlar) har 10 qadamda yozilishini bildiradi.
# report_to='none': O'qitish loglarini hech qanday tashqi monitoring platformasiga yubormaslikni ko'rsatadi.
# trainer = Trainer(...): Bu qator Trainer obyektini yaratadi, bu ob'ekt modelni o'qitishni boshqaradi. Unga quyidagilar beriladi:
# model=model: O'qitilishi kerak bo'lgan model (bizning holatimizda AutoModelForSequenceClassification orqali yuklangan model).
# args=training_args: Yuqorida belgilangan o'qitish parametrlari.
# train_dataset=tokenized_train: O'qitish uchun ishlatiladigan ma'lumotlar to'plami.
# eval_dataset=tokenized_test: Baholash uchun ishlatiladigan ma'lumotlar to'plami.
# trainer.train(): Bu funksiya modelni yuqorida ko'rsatilgan parametrlarga asosan o'qitish jarayonini boshlaydi.

Step,Training Loss
10,0.223447
20,0.010415
30,0.003087
40,0.001607
50,0.001027
60,0.000770
70,0.000589
80,0.000492
90,0.000422
100,0.000332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.012919334567240662, metrics={'train_runtime': 85.2112, 'train_samples_per_second': 35.207, 'train_steps_per_second': 2.218, 'total_flos': 198701097984000.0, 'train_loss': 0.012919334567240662, 'epoch': 3.0})

## **Modelni sinab ko'rish**

### O'qitilgan modelimizni yangi kinofilm sharhi bilan sinab ko'ramiz va u qanday kayfiyatni (ijobiy yoki salbiy) bashorat qilishini tekshiramiz.

In [ ]:
import torch

def predict_sentiment(text, model, tokenizer):
    # Matnni tokenizatsiya qilish
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=256, return_tensors="pt")

    # Inputlarni model joylashgan qurilmaga o'tkazish
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Modelga kiritish va bashorat qilish
    with torch.no_grad(): # Gradientlarni hisoblashni o'chirib qo'yamiz, chunki biz o'qitmayapmiz
        outputs = model(**inputs)

    # Logits-dan eng yuqori ehtimollikni olish
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()

    # Bashoratni talqin qilish
    if prediction == 0:
        return "Salbiy (Negative)"
    else:
        return "Ijobiy (Positive)"

# Test uchun yangi kinofilm sharhi
review1 = "This movie was absolutely fantastic! I loved every minute of it."
review2 = "This film was a complete waste of time. The acting was terrible and the plot was boring."
review3 = "This film was very boring and not interested"

print(f"Sharh: '{review1}' -> Bashorat: {predict_sentiment(review1, model, tokenizer)}")
print(f"Sharh: '{review2}' -> Bashorat: {predict_sentiment(review2, model, tokenizer)}")
print(f"Sharh: '{review3}' -> Bashorat: {predict_sentiment(review2, model, tokenizer)}")

Sharh: 'This movie was absolutely fantastic! I loved every minute of it.' -> Bashorat: Ijobiy (Positive)
Sharh: 'This film was a complete waste of time. The acting was terrible and the plot was boring.' -> Bashorat: Ijobiy (Positive)
Sharh: 'This film was very boring and not interested' -> Bashorat: Ijobiy (Positive)


## **Model aniqligini baholash**


### Model aniqligi deyarli 100% chiqmoqda yani bunda overfitting muammosi sodir bo'ldi modelga haddan tashqari moslashib ketdi ammo yangi malumotlarni yaxsh tahlil qila olmayabdi chunki bunga sabab taest malumotlarini juda kam miqdorda olgan edik test va train malumotlari qancha katta bo'lsa model aniqligi ham oshadi

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Trainer metrikasini yangilash
trainer.compute_metrics = compute_metrics

# Notebook progress xatosini chetlab o'tish uchun callback'larni vaqtincha tozalaymiz
old_callbacks = trainer.callback_handler.callbacks.copy()
trainer.callback_handler.callbacks = [cb for cb in trainer.callback_handler.callbacks if "NotebookProgressCallback" not in str(type(cb))]

# Modelni baholash
try:
    results = trainer.evaluate(eval_dataset=tokenized_test)
    print("Model baholash natijalari:")
    for key, value in results.items():
        print(f"{key}: {value:.4f}")
finally:
    # Callback'larni qayta tiklaymiz
    trainer.callback_handler.callbacks = old_callbacks

Model baholash natijalari:
eval_loss: 0.0004
eval_accuracy: 1.0000
eval_f1: 1.0000
eval_precision: 1.0000
eval_recall: 1.0000
eval_runtime: 2.9383
eval_samples_per_second: 102.1010
eval_steps_per_second: 6.4660
epoch: 3.0000
